# 3. 데이터 시각화 — Matplotlib

> **제3장** · **이론편 대응: 4.4절 (SVD와 차원 축소), 5.4절 (경사하강법)**
> **예상 소요**: 40분
> **필요 사양**: CPU만으로 충분

---

## 이 장에서 하는 일

숫자만 봐서는 알기 어려운 것들을 그림으로 확인한다.

| 절 | 그리는 것 | 이론편 대응 |
|---|---|---|
| 2 | 기본 그래프 (선·산점도·막대) | — |
| 3 | 손실 곡선 읽기 | 11.6절 |
| 4 | **PCA 주성분 시각화** | 4.4절 |
| 5 | 경사하강 궤적 | 5.4절 |
| 6 | 여러 그림 배치 | — |

시각화는 결과를 남에게 보여주기 위한 것만이 아니다. **학습이 제대로 되고 있는지 판단하는 도구**다.
이론편 11.6절에서 손실 곡선 세 가지 패턴을 봤는데, 그것을 직접 그려 본다.

---

## 1. 한글 폰트 설정 — 먼저 해결하고 가자

Matplotlib은 기본 상태에서 **한글을 네모(□□□)로 표시한다.** 그래프에 한글 제목을 넣으려면
폰트를 지정해야 하는데, 이 문제로 시간을 뺏기는 경우가 많으므로 여기서 정리한다.

**Windows에는 '맑은 고딕'이 기본 설치되어 있으므로** 대부분 아래 셀 한 번으로 해결된다.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

print("=" * 50)
print("한글 폰트 설정")
print("=" * 50)

system = platform.system()
print(f"운영체제: {system}")

# 후보 폰트 (운영체제별)
candidates = {
    "Windows": ["Malgun Gothic", "NanumGothic", "Gulim"],
    "Darwin":  ["AppleGothic", "NanumGothic"],
    "Linux":   ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"],
}

available = {f.name for f in fm.fontManager.ttflist}
chosen = None

for name in candidates.get(system, []):
    if name in available:
        chosen = name
        break

if chosen:
    plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False   # 마이너스 기호 깨짐 방지
    print(f"[OK] 폰트 설정: {chosen}")
else:
    print("[주의] 한글 폰트를 찾지 못했습니다.")
    print("  그래프의 한글이 네모로 보일 수 있습니다.")
    print("  Windows라면 '맑은 고딕'이 기본 설치되어 있어야 합니다.")
    print("  해결이 어렵다면 그래프 라벨을 영문으로 쓰세요.")

print()
print("설치된 한글 계열 폰트:")
korean_fonts = sorted({f for f in available
                       if any(k in f for k in ["Malgun", "Nanum", "Gothic", "CJK", "Gulim", "Batang"])})
for f in korean_fonts[:10]:
    print(f"  - {f}")
if not korean_fonts:
    print("  (없음)")

### `axes.unicode_minus = False`가 필요한 이유

한글 폰트로 바꾸면 **음수 부호가 깨지는** 부작용이 생긴다. Matplotlib이 기본으로 쓰는 유니코드
마이너스 기호(−, U+2212)가 한글 폰트에 없는 경우가 있기 때문이다.

이 설정은 대신 일반 하이픈(-)을 쓰게 해서 문제를 피한다. 한글 폰트를 지정할 때는
**항상 이 줄을 함께 넣는 습관**을 들이는 것이 좋다.

---

## 2. 기본 그래프 세 가지

가장 많이 쓰는 세 종류를 익힌다. 이후 모든 실습에서 이 형태가 반복된다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- (1) 선 그래프: 연속적인 변화 ---
x = np.linspace(0, 10, 100)
axes[0].plot(x, np.sin(x), label="sin(x)", linewidth=2)
axes[0].plot(x, np.cos(x), label="cos(x)", linewidth=2, linestyle="--")
axes[0].set_title("선 그래프 - 연속 변화")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- (2) 산점도: 두 값의 관계 ---
rng = np.random.RandomState(0)
n = 100
height = rng.normal(170, 8, n)
weight = height * 0.7 - 55 + rng.normal(0, 4, n)
axes[1].scatter(height, weight, alpha=0.6, s=30)
axes[1].set_title("산점도 - 두 값의 관계")
axes[1].set_xlabel("키 (cm)")
axes[1].set_ylabel("몸무게 (kg)")
axes[1].grid(alpha=0.3)

# --- (3) 막대 그래프: 범주별 비교 ---
categories = ["A", "B", "C", "D"]
values = [23, 45, 31, 38]
axes[2].bar(categories, values, color=["#1E40AF", "#EA580C", "#0D9488", "#7C3AED"])
axes[2].set_title("막대 그래프 - 범주 비교")
axes[2].set_ylabel("값")
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("각 그래프가 쓰이는 곳")
print("  선 그래프 : 손실 곡선, 학습률 변화 (거의 모든 실습)")
print("  산점도    : 데이터 분포 (08장), 잠재 공간 (18장)")
print("  막대      : 성능 비교, 어텐션 가중치 (17장)")

### `fig, ax` 방식을 쓰는 이유

Matplotlib에는 두 가지 사용법이 있다.

```python
# 방식 1: plt에 직접 (간단하지만 그림이 여러 개면 헷갈림)
plt.plot(x, y)
plt.title("제목")

# 방식 2: fig, ax 를 만들어서 (권장)
fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_title("제목")
```

이 책은 **방식 2**를 쓴다. 그림을 여러 개 배치할 때 어느 그림에 무엇을 그리는지 명확하고,
나중에 그림을 파일로 저장하거나 조정하기도 쉽기 때문이다.

메서드 이름이 조금 달라지는 점만 주의하면 된다 — `plt.title()` → `ax.set_title()`처럼
대부분 `set_` 접두어가 붙는다.

---

## 3. 손실 곡선 읽기 — 이론편 11.6절

이론편 11.6절에서 손실 곡선의 세 가지 패턴을 봤다. 실제로 그려 보면서
**무엇을 보고 판단하는지** 익힌다.

핵심은 **훈련 손실과 검증 손실을 함께 그리는 것**이다. 하나만 봐서는 과대적합을 알 수 없다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epochs = np.arange(1, 31)

# 세 가지 학습 상황을 흉내 낸 값
patterns = {
    "정상적인 학습": {
        "train": 2.0 * 0.90**epochs + 0.15,
        "val":   2.0 * 0.90**epochs + 0.22,
        "note":  "둘 다 내려가고 격차가 작다",
    },
    "과대적합": {
        "train": 2.0 * 0.85**epochs + 0.05,
        "val":   np.concatenate([
                    2.0 * 0.85**np.arange(1, 11) + 0.15,
                    2.0 * 0.85**10 + 0.15 + np.arange(1, 21) * 0.045]),
        "note":  "검증 손실이 다시 올라간다",
    },
    "학습률이 너무 큼": {
        "train": 2.0 + 0.6 * np.sin(epochs * 1.6),
        "val":   2.1 + 0.6 * np.sin(epochs * 1.6 + 0.3),
        "note":  "진동하며 내려가지 않는다",
    },
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (title, d) in zip(axes, patterns.items()):
    ax.plot(epochs, d["train"], label="훈련 손실", linewidth=2)
    ax.plot(epochs, d["val"], label="검증 손실", linewidth=2, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("에폭")
    ax.set_ylim(0, 3)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.text(0.5, -0.25, d["note"], transform=ax.transAxes,
            ha="center", fontsize=9)

axes[0].set_ylabel("손실")

# 과대적합 그림에 조기 종료 지점 표시
axes[1].axvline(10, color="red", linestyle=":", linewidth=1.5)
axes[1].text(11, 2.4, "여기서 멈춰야", color="red", fontsize=9)

plt.tight_layout()
plt.show()

### 판단 기준 정리

| 관찰 | 진단 | 대응 |
|---|---|---|
| 둘 다 완만히 하락, 격차 작음 | 정상 | 계속 학습 |
| 훈련은 하락, 검증은 상승 | 과대적합 | 조기 종료 / 정규화 / 데이터 추가 |
| 둘 다 진동, 하락 없음 | 학습률 과다 | 학습률 축소 |
| 둘 다 거의 변화 없음 | 학습률 과소 또는 모델 부족 | 학습률 증대 / 모델 확대 |
| 훈련 손실이 NaN | 발산 | 학습률 대폭 축소, 데이터 확인 |

앞으로 모델을 학습시킬 때마다 이 곡선을 그려 보게 된다. **숫자만 보지 말고 그림으로 확인하는 습관**이
문제를 빨리 찾는 데 도움이 된다.

---

## 4. PCA 주성분 시각화 — 이론편 4.4절 ★

이론편 4.4절 그림 4-4에서 "데이터가 퍼진 방향"을 찾아 차원을 줄이는 과정을 봤다.
이번에는 직접 계산하고 그려 본다.

절차는 다음과 같다.

1. 데이터의 평균을 빼서 중심을 원점으로 옮긴다
2. 공분산 행렬을 구한다
3. **공분산 행렬의 고유벡터**가 주성분 방향이다 (이론편 4.3~4.4절)
4. 고유값이 큰 방향이 데이터가 많이 퍼진 방향이다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1) 대각선 방향으로 퍼진 데이터 만들기 ---
rng = np.random.RandomState(42)
n = 200
x1 = rng.randn(n) * 2.0
x2 = x1 * 0.8 + rng.randn(n) * 0.5
X = np.stack([x1, x2], axis=1)

# --- 2) 중심을 원점으로 ---
X_centered = X - X.mean(axis=0)

# --- 3) 공분산 행렬과 고유분해 ---
cov = np.cov(X_centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov)

# 고유값이 큰 순으로 정렬
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("=" * 50)
print("PCA 계산 결과")
print("=" * 50)
print(f"공분산 행렬:\n{cov.round(4)}")
print()
print(f"고유값        : {eigenvalues.round(4)}")
print(f"설명 분산 비율 : {(eigenvalues / eigenvalues.sum() * 100).round(2)} %")
print()
print(f"제1주성분 방향 : {eigenvectors[:, 0].round(4)}")
print(f"제2주성분 방향 : {eigenvectors[:, 1].round(4)}")
print()
print(f"→ 첫 방향 하나가 전체 분산의 {eigenvalues[0]/eigenvalues.sum()*100:.1f}%를 설명한다")
print("  (이론편 4.4절에서 '약 97%'라고 한 값)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- 왼쪽: 원본 데이터와 주성분 방향 ---
ax = axes[0]
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, s=25, color="#64748B")

# 주성분을 화살표로 (길이는 고유값의 제곱근에 비례)
colors = ["#EA580C", "#0D9488"]
labels = ["제1주성분", "제2주성분"]
for i in range(2):
    vec = eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 2.5
    ax.arrow(0, 0, vec[0], vec[1], head_width=0.25, head_length=0.3,
             fc=colors[i], ec=colors[i], linewidth=2.5, zorder=5)
    ax.text(vec[0] * 1.15, vec[1] * 1.15, labels[i],
            color=colors[i], fontsize=10, ha="center")

ax.set_title("원본 데이터와 주성분 방향")
ax.set_xlabel("특성 1")
ax.set_ylabel("특성 2")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)

# --- 오른쪽: 제1주성분으로 투영 (2차원 → 1차원) ---
ax = axes[1]
pc1 = eigenvectors[:, 0]
projected = X_centered @ pc1                    # 1차원 좌표
reconstructed = np.outer(projected, pc1)        # 다시 2차원으로 되돌린 위치

ax.scatter(X_centered[:, 0], X_centered[:, 1],
           alpha=0.25, s=25, color="#94A3B8", label="원본")
ax.scatter(reconstructed[:, 0], reconstructed[:, 1],
           alpha=0.7, s=25, color="#EA580C", label="투영 결과")

# 원본과 투영점을 잇는 선 (일부만)
for i in range(0, n, 8):
    ax.plot([X_centered[i, 0], reconstructed[i, 0]],
            [X_centered[i, 1], reconstructed[i, 1]],
            color="#CBD5E1", linewidth=0.7, zorder=1)

ax.set_title(f"제1주성분에 투영 (정보 {eigenvalues[0]/eigenvalues.sum()*100:.1f}% 보존)")
ax.set_xlabel("특성 1")
ax.set_ylabel("특성 2")
ax.set_aspect("equal")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"2차원 → 1차원으로 줄였는데 정보의 {eigenvalues[0]/eigenvalues.sum()*100:.1f}%가 남았다.")
print("회색 선은 투영하면서 버려진 부분(제2주성분 성분)이다.")

### SVD로도 같은 결과가 나온다

이론편 4.4절에서 "PCA의 주성분은 SVD로 분해했을 때 나오는 성분과 같다"고 했다.
직접 확인해 보자.

In [ ]:
import numpy as np

# 고유분해 방식 (위에서 계산한 것)
print("=" * 50)
print("고유분해 vs SVD")
print("=" * 50)
print(f"[고유분해] 제1주성분 : {eigenvectors[:, 0].round(4)}")
print(f"[고유분해] 설명 비율 : {(eigenvalues / eigenvalues.sum() * 100).round(2)} %")
print()

# SVD 방식
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
svd_ratio = S**2 / (S**2).sum() * 100

print(f"[SVD]     제1주성분 : {Vt[0].round(4)}")
print(f"[SVD]     설명 비율 : {svd_ratio.round(2)} %")
print()

same_dir = np.allclose(np.abs(eigenvectors[:, 0]), np.abs(Vt[0]))
same_ratio = np.allclose(eigenvalues / eigenvalues.sum(), S**2 / (S**2).sum())

print(f"방향 일치(부호 무시) : {same_dir}")
print(f"설명 비율 일치       : {same_ratio}")
assert same_dir and same_ratio
print()
print("[OK] 이론편 4.4절 서술과 일치 — 두 방법은 같은 것을 계산한다")
print()
print("※ 부호가 반대로 나올 수 있는데, 벡터의 방향만 뒤집힌 것이라 의미는 같다.")

---

## 5. 경사하강 궤적 — 이론편 5.4절

이론편 5.4절에서 $L(w) = (w-3)^2$을 여섯 걸음 걸어 최솟값에 다가가는 과정을 표로 봤다.
그것을 그림으로 그려 본다.

손계산 값은 다음과 같았다.

| 스텝 | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| w | 5.000 | 3.800 | 3.320 | 3.128 | 3.051 | 3.021 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 이론편 5.4절과 같은 설정
def L(w):
    return (w - 3) ** 2

def grad(w):
    return 2 * (w - 3)

w = 5.0
eta = 0.3
history = [w]

for _ in range(6):
    w = w - eta * grad(w)
    history.append(w)

history = np.array(history)

print("=" * 50)
print("경사하강 궤적 (이론편 5.4절 검증)")
print("=" * 50)
print(f"{'스텝':<6}{'w':<12}{'손실':<12}{'그래디언트'}")
print("-" * 50)
for i, wi in enumerate(history[:6]):
    print(f"{i:<6}{wi:<12.4f}{L(wi):<12.4f}{grad(wi):.4f}")

expected = [5.0, 3.8, 3.32, 3.128, 3.0512, 3.0205]
assert np.allclose(history[:6], expected, atol=1e-3), "이론편 값과 다릅니다"
print("-" * 50)
print("[OK] 이론편 5.4절 손계산과 일치")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- 왼쪽: 손실 곡선 위의 궤적 ---
ax = axes[0]
w_range = np.linspace(1.5, 5.5, 200)
ax.plot(w_range, L(w_range), color="#1E40AF", linewidth=2, label="L(w) = (w-3)²")

# 각 스텝을 점과 화살표로
ax.scatter(history, L(history), color="#EA580C", s=70, zorder=5, label="각 스텝")
for i in range(len(history) - 1):
    ax.annotate("", xy=(history[i+1], L(history[i+1])),
                xytext=(history[i], L(history[i])),
                arrowprops=dict(arrowstyle="->", color="#EA580C", lw=1.5))

ax.scatter([3], [0], color="#0D9488", s=120, marker="*", zorder=6, label="최솟값")
ax.set_title("손실 곡선 위의 이동 경로")
ax.set_xlabel("w")
ax.set_ylabel("손실 L(w)")
ax.legend()
ax.grid(alpha=0.3)

# --- 오른쪽: 스텝별 값 변화 ---
ax = axes[1]
steps = np.arange(len(history))
ax.plot(steps, history, marker="o", color="#EA580C", linewidth=2, label="w")
ax.axhline(3, color="#0D9488", linestyle="--", linewidth=1.5, label="목표값 3")
ax.set_title("스텝별 w의 변화")
ax.set_xlabel("스텝")
ax.set_ylabel("w")
ax.legend()
ax.grid(alpha=0.3)

# 이동량이 줄어드는 것 표시
for i in range(min(3, len(history)-1)):
    move = abs(history[i+1] - history[i])
    ax.annotate(f"{move:.2f}", xy=(i + 0.5, (history[i] + history[i+1]) / 2),
                fontsize=8, color="#64748B")

plt.tight_layout()
plt.show()

print("오른쪽 그림의 숫자는 한 걸음의 이동량이다.")
print("최솟값에 가까워질수록 저절로 작아진다 — 그래디언트가 작아지기 때문이다.")

---

## 6. 여러 그림을 함께 배치하기

앞에서 계속 써 온 `plt.subplots(행, 열)`을 정리한다. 실습에서 결과를 비교할 때 자주 쓴다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 2행 3열 배치
fig, axes = plt.subplots(2, 3, figsize=(13, 6))

x = np.linspace(0, 2*np.pi, 100)
functions = [
    ("sin(x)",      np.sin(x)),
    ("cos(x)",      np.cos(x)),
    ("sin(2x)",     np.sin(2*x)),
    ("x²/10",       x**2 / 10),
    ("exp(-x)",     np.exp(-x)),
    ("sin(x)/x",    np.sin(x) / np.where(x == 0, 1e-9, x)),
]

for ax, (name, y) in zip(axes.flat, functions):
    ax.plot(x, y, linewidth=2)
    ax.set_title(name)
    ax.grid(alpha=0.3)

fig.suptitle("2행 3열 배치 예시", fontsize=13)
plt.tight_layout()
plt.show()

print("axes.flat 을 쓰면 2차원 배열을 1차원처럼 순회할 수 있다.")
print("axes[0, 1] 처럼 위치를 직접 지정할 수도 있다.")

### 그림을 파일로 저장하기

발표 자료나 보고서에 넣으려면 파일로 저장해야 한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 저장 폴더 준비
out_dir = Path.cwd()
if out_dir.name.startswith("part"):
    out_dir = out_dir.parent
out_dir = out_dir / "outputs"
out_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), linewidth=2)
ax.set_title("저장 예시")
ax.grid(alpha=0.3)

save_path = out_dir / "sample_plot.png"
fig.savefig(save_path,
            dpi=150,              # 해상도 (인쇄용은 200~300)
            bbox_inches="tight",  # 여백 잘라내기
            facecolor="white")    # 배경색 (기본은 투명)
plt.close(fig)

print(f"저장 완료: {save_path}")
print(f"파일 크기: {save_path.stat().st_size / 1024:.1f} KB")
print()
print("주요 옵션")
print("  dpi          : 해상도. 화면용 100, 인쇄용 200~300")
print("  bbox_inches  : 'tight' 로 하면 불필요한 여백 제거")
print("  facecolor    : 지정하지 않으면 배경이 투명해질 수 있음")

---

## 7. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 4.4 | PCA 설명 분산비 약 97% | 97.67% ✓ |
| 4.4 | 고유분해 = SVD | 일치 확인 ✓ |
| 5.4 | 경사하강 6스텝 궤적 | 완전 일치 ✓ |
| 11.6 | 손실 곡선 세 패턴 | 그림으로 확인 ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 한글 폰트 | `font.family` 지정 + `axes.unicode_minus = False` |
| 그리기 방식 | `fig, ax = plt.subplots()` 방식 권장 |
| 손실 곡선 | **훈련과 검증을 함께** 그려야 과대적합이 보임 |
| PCA | 공분산 행렬의 고유벡터 = 주성분 방향 |
| 저장 | `bbox_inches="tight"`, `facecolor="white"` |

### 다음 장

**4. 선형회귀 직접 구현 — NumPy만으로** — 이론편 5장의 경사하강법으로 실제 데이터에 직선을 맞춘다.
NumPy만으로 구현하고, 학습 과정을 이 장에서 배운 방법으로 시각화한다.